# 7.3 K-Means from Scratch

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 7: Clustering Techniques

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-07-clustering-techniques/07.03_k-means-from-scratch.ipynb)

Companion code for Section 7.3 of the book. Run the cells in order from top to bottom.

**Example 7.3.1**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score

class KMeansScratch:
    '''
    K-Means clustering from scratch.
    Supports both random and K-Means++ initialisation.
    '''

    def __init__(self, k, init='kmeans++', max_iter=300, tol=1e-4, n_init=10, random_state=None):
        self.k            = k
        self.init         = init
        self.max_iter     = max_iter
        self.tol          = tol
        self.n_init       = n_init
        self.rng          = np.random.RandomState(random_state)
        self.centroids_   = None
        self.labels_      = None
        self.inertia_     = np.inf
        self.n_iter_      = 0
        self.history_     = []   # Store centroids at each iteration

    def _init_random(self, X):
        '''Random initialisation: choose k random data points.'''
        idx = self.rng.choice(len(X), self.k, replace=False)
        return X[idx].copy()

    def _init_kmeans_plus_plus(self, X):
        '''K-Means++ initialisation: spread centroids across the data.'''
        n = len(X)
        # Choose first centroid uniformly at random
        centroids = [X[self.rng.randint(n)]]

        for _ in range(1, self.k):
            # Compute squared distance from each point to nearest existing centroid
            D2 = np.array([
                min(np.sum((x - c)**2) for c in centroids)
                for x in X
            ])
            # Sample next centroid with probability proportional to D²
            probs    = D2 / D2.sum()
            cum_prob = np.cumsum(probs)
            r = self.rng.rand()
            idx = np.searchsorted(cum_prob, r)
            centroids.append(X[idx])

        return np.array(centroids)

    def _assign_clusters(self, X, centroids):
        '''Assign each point to its nearest centroid.'''
        # Compute distances: shape (n_samples, k)
        distances = np.array([
            np.sum((X - c)**2, axis=1)   # Squared Euclidean
            for c in centroids
        ]).T   # Shape: (n_samples, k)
        return np.argmin(distances, axis=1)

    def _update_centroids(self, X, labels):
        '''Update centroids to mean of assigned points.'''
        centroids = np.zeros((self.k, X.shape[1]))
        for j in range(self.k):
            mask = labels == j
            if mask.sum() > 0:
                centroids[j] = X[mask].mean(axis=0)
            else:
                # Empty cluster: re-initialise to random point
                centroids[j] = X[self.rng.randint(len(X))]
        return centroids

    def _compute_inertia(self, X, labels, centroids):
        '''Compute WCSS (within-cluster sum of squares).'''
        inertia = 0.0
        for j in range(self.k):
            mask = labels == j
            if mask.sum() > 0:
                inertia += np.sum((X[mask] - centroids[j])**2)
        return inertia

    def _fit_single(self, X):
        '''Single run of K-Means.'''
        # Initialise centroids
        if self.init == 'kmeans++':
            centroids = self._init_kmeans_plus_plus(X)
        else:
            centroids = self._init_random(X)

        history = [centroids.copy()]

        for iteration in range(self.max_iter):
            # Assignment step
            labels = self._assign_clusters(X, centroids)

            # Update step
            new_centroids = self._update_centroids(X, labels)

            # Store history
            history.append(new_centroids.copy())

            # Check convergence
            shift = np.max(np.linalg.norm(new_centroids - centroids, axis=1))
            centroids = new_centroids

            if shift < self.tol:
                break

        inertia = self._compute_inertia(X, labels, centroids)
        return centroids, labels, inertia, iteration+1, history

    def fit(self, X):
        '''Fit K-Means with n_init restarts, keep best result.'''
        best_inertia = np.inf

        for run in range(self.n_init):
            centroids, labels, inertia, n_iter, history = self._fit_single(X)
            if inertia < best_inertia:
                best_inertia       = inertia
                self.centroids_    = centroids
                self.labels_       = labels
                self.inertia_      = inertia
                self.n_iter_       = n_iter
                self.history_      = history

        return self

    def predict(self, X):
        return self._assign_clusters(X, self.centroids_)

    def fit_predict(self, X):
        return self.fit(X).labels_


# ── Test on synthetic data ────────────────────────────────────────
np.random.seed(42)
X, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.7, random_state=42)
X = StandardScaler().fit_transform(X)

# Fit from scratch
km = KMeansScratch(k=4, init='kmeans++', n_init=10, random_state=42)
km.fit(X)

print('=== K-Means from Scratch ===')
print(f'Inertia:          {km.inertia_:.4f}')
print(f'Iterations:       {km.n_iter_}')
print(f'Cluster sizes:    {dict(zip(*np.unique(km.labels_, return_counts=True)))}')
print(f'\nFinal centroids:')
for i, c in enumerate(km.centroids_):
    print(f'  C{i}: [{c[0]:+.4f}, {c[1]:+.4f}]')

# Compare with sklearn
from sklearn.cluster import KMeans
km_sk = KMeans(n_clusters=4, init='k-means++', n_init=10, random_state=42)
km_sk.fit(X)
print(f'\nScikit-learn inertia: {km_sk.inertia_:.4f}')
print(f'Our inertia:          {km.inertia_:.4f}')
print(f'Match: {np.allclose(km.inertia_, km_sk.inertia_, rtol=0.01)}')

# Adjusted Rand Index (how similar are our labels to sklearn's?)
ari = adjusted_rand_score(km_sk.labels_, km.labels_)
print(f'Adjusted Rand Index (our vs sklearn): {ari:.4f}  (1.0 = identical)')

# ── Visualise convergence animation frames ────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('K-Means Convergence - Step by Step', fontsize=14, fontweight='bold')
palette = ['#E53935','#1976D2','#43A047','#FF6F00']

# Re-run with tracking for a single run
km_track = KMeansScratch(k=4, init='kmeans++', n_init=1, random_state=7)
km_track.fit(X)

show_steps = [0, 1, 2, 3, 4, 5, 6, km_track.n_iter_]
for idx, step in enumerate(show_steps):
    ax = axes[idx//4, idx%4]
    step = min(step, len(km_track.history_)-1)
    centroids_step = km_track.history_[step]

    # Assign labels for this step
    labels_step = km_track._assign_clusters(X, centroids_step)

    for k in range(4):
        mask = labels_step == k
        ax.scatter(X[mask,0], X[mask,1], s=12, alpha=0.6, color=palette[k])
    ax.scatter(centroids_step[:,0], centroids_step[:,1],
              s=200, c='black', marker='*', zorder=10, edgecolors='white', lw=1.5)

    title = 'Initial centroids' if step==0 else f'Iteration {step}'
    if step == km_track.n_iter_: title += ' (Converged)'
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('kmeans_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nConvergence animation saved.')